# Tag Extraction

This notebook belongs to **AISTER Step 2 (Image-based analysis)** and extracts structured tags from image captions.

Prepared by **Web2Learn** for the **AISTER - Krovets collection** project (2026-03-26).

**Inputs and outputs**
- Input: `captions.csv` generated by `02_image_captioning.ipynb`.
- Output: `tags.csv` containing structured fields (`figures`, `attire`, `objects`, `background`, `scenes`, `text`, `damage`).

**How to run**
Run this notebook sequentially from top to bottom.

## Setup

Install required dependencies.

In [1]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Imports

Import all libraries used in this notebook.

In [2]:
from pathlib import Path
import json
import time

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from google.colab import drive

Mount Google Drive to access captions CSV and save results.

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

Set all user-editable constants in the next code cell before running the model and batch extraction.

In [6]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

# Strategy for placing model layers on available hardware.
DEVICE_MAP = "auto"

# Memory budget for automatic placement during model loading.
GPU_MAX_MEMORY = "14GiB"  # max VRAM to use on GPU 0
CPU_MAX_MEMORY = "12GiB"  # max system RAM fallback for offloaded weights

# Token limits for prompt+caption input and generated JSON output.
MAX_INPUT_TOKENS = 500
MAX_NEW_TOKENS = 500

# Input/output paths for batch extraction.
CAPTIONS_CSV = "/content/drive/MyDrive/W2L/Projects/AISTER/w2l/step 2/outputs/captions.csv"
OUTPUT_CSV = "/content/drive/MyDrive/W2L/Projects/AISTER/w2l/step 2/outputs/tags.csv"

## Model

Load the text-only Qwen model and tokenizer.

In [7]:
# Set a fixed seed for reproducible model behavior across runs.
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

has_cuda = torch.cuda.is_available()

# Load tokenizer for the selected text model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the text model with selected precision/device settings.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE_MAP if has_cuda else "cpu",
    torch_dtype=torch.float16 if has_cuda else torch.float32,
    max_memory={0: GPU_MAX_MEMORY, "cpu": CPU_MAX_MEMORY} if has_cuda else None,
    low_cpu_mem_usage=True,
)
model.eval()

print(f"Loaded: {MODEL_NAME}")
print(f"CUDA: {has_cuda}")
print(f"device_map: {getattr(model, 'hf_device_map', None)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3.5-4B
CUDA: True
device_map: None


## Prompt

The prompt instructs the model to extract structured tags from a free-text caption.
The `<<<CAPTION>>>` placeholder is replaced with the actual caption at runtime.

In [8]:
PROMPT = """
Return one valid JSON object only.

Caption:
<<<CAPTION>>>

Schema:
{
  "figures": [],
  "attire": [],
  "objects": [],
  "background": [],
  "scenes": [],
  "text": [],
  "damage": []
}

Rules:
- Use only information explicitly in the caption.
- Use short noun phrases.
- Do not invent or explain.
- No text before or after the JSON.
- If no items, use [].

Meanings:
- figures = human persons only
- attire = clothing and garment features
- objects = physical non-wearable things
- background = backdrop, surface, environment
- scenes = depiction type or genre
- text = quoted text, watermark, or explicit text-visibility phrase
- damage = damage or condition phrases
"""

## Tag extraction

Send a batch of captions through the model, then parse the JSON output into tag fields.

In [9]:
TAG_FIELDS = ["figures", "attire", "objects", "background", "scenes", "text", "damage"]


def generate_tags(
    captions: list[str],
    model=model,
    tokenizer=tokenizer,
) -> list[dict]:
    """Generate structured tags for a batch of captions.

    Args:
        captions: List of caption strings to process.
        model: Loaded text model used for generation.
        tokenizer: Matching tokenizer for prompt formatting and decoding.
    """
    messages_batch = [
        [
            {"role": "system", "content": "You are a data extraction AI. Output only valid JSON."},
            {"role": "user", "content": PROMPT.replace("<<<CAPTION>>>", cap)},
        ]
        for cap in captions
    ]

    texts = [
        tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False) + "{"
        for m in messages_batch
    ]

    inputs = tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INPUT_TOKENS,
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    results = []
    for i in range(len(captions)):
        decoded = tokenizer.decode(gen[i, prompt_len:], skip_special_tokens=True).strip()
        raw = "{" + decoded
        results.append(_parse_tags(raw))

    return results


def _parse_tags(text: str) -> dict:
    """Extract the first JSON object from text and return parsed tag dict."""
    empty = {k: [] for k in TAG_FIELDS}

    start = text.find("{")
    if start == -1:
        return empty

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    data = json.loads(text[start:i + 1])
                    return {k: data.get(k, []) if isinstance(data.get(k), list) else [] for k in TAG_FIELDS}
                except json.JSONDecodeError:
                    return empty

    return empty

### Example usage

Try extracting tags from a single caption.

In [11]:
example_caption = (
    "A painting depicts a man in a dark blue military uniform and a black hat, "
    "riding a white horse with a red saddle. A woman in a red and blue "
    "embroidered dress stands beside him, holding a small wooden bucket. "
    "In the background, there are houses, a windmill, trees, and a cloudy sky. "
    "The scene is set in a rural landscape. There is text at the bottom in "
    "Ukrainian: 'Дівчино Моя, на піжке коня, срубленої нової кріпниці, споного "
    "відра!' and a watermark 'Сушия колекція' in the lower left corner. "
    "The painting shows visible signs of age and wear."
)

tags = generate_tags([example_caption])[0]
print(json.dumps(tags, indent=2))

{
  "figures": [
    "man",
    "woman"
  ],
  "attire": [
    "dark blue military uniform",
    "black hat",
    "red and blue embroidered dress"
  ],
  "objects": [
    "white horse",
    "red saddle",
    "small wooden bucket"
  ],
  "background": [
    "houses",
    "windmill",
    "trees",
    "cloudy sky"
  ],
  "scenes": [
    "rural landscape"
  ],
  "text": [
    "\u0414\u0456\u0432\u0447\u0438\u043d\u043e \u041c\u043e\u044f, \u043d\u0430 \u043f\u0456\u0436\u043a\u0435 \u043a\u043e\u043d\u044f, \u0441\u0440\u0443\u0431\u043b\u0435\u043d\u043e\u0457 \u043d\u043e\u0432\u043e\u0457 \u043a\u0440\u0456\u043f\u043d\u0438\u0446\u0456, \u0441\u043f\u043e\u043d\u043e\u0433\u043e \u0432\u0456\u0434\u0440\u0430!",
    "\u0421\u0443\u0448\u0438\u044f \u043a\u043e\u043b\u0435\u043a\u0446\u0456\u044f"
  ],
  "damage": [
    "visible signs of age and wear"
  ]
}


### Batch extraction

The next code cell loads captions from `CAPTIONS_CSV`, runs tag extraction in batches, and writes results to `OUTPUT_CSV` incrementally so progress is preserved.

Paths come from the `Configuration` cell. You can also tune `batch_size`, `resume`, and `limit` in the run cell.

In [12]:
def batch_extraction(
    captions_csv: str,
    output_csv: str,
    model=model,
    tokenizer=tokenizer,
    batch_size: int = 8,
    limit: int | None = None,
    resume: bool = True
) -> pd.DataFrame:
    """Extract tags from captions CSV and save to output CSV.

    Args:
        captions_csv: Path to input captions CSV.
        output_csv: Path to output tags CSV.
        model: Loaded text model used for generation.
        tokenizer: Matching tokenizer for prompt formatting and decoding.
        batch_size: Number of captions processed per model call.
        limit: Optional cap on how many pending captions to process in this run.
        resume: If True, skip image_ids already present in output CSV.
    """
    captions_df = pd.read_csv(captions_csv)
    captions_df["image_id"] = captions_df["image_id"].astype(str)

    processed_ids: set[str] = set()
    existing_df = pd.DataFrame()

    if resume and Path(output_csv).exists():
        existing_df = pd.read_csv(output_csv)
        if "image_id" in existing_df.columns:
            processed_ids = set(existing_df["image_id"].astype(str))
        print(f"Resuming: {len(processed_ids)} already processed.")

    remaining = captions_df[~captions_df["image_id"].isin(processed_ids)].copy()
    if limit is not None:
        remaining = remaining.head(limit)

    total = len(remaining)
    print(f"Processing {total} captions (batch_size={batch_size}).\n")

    t0 = time.perf_counter()

    for start in range(0, total, batch_size):
        batch = remaining.iloc[start:start + batch_size]
        tags_list = generate_tags(
            batch["caption"].tolist(),
            model=model,
            tokenizer=tokenizer,
        )

        rows = []
        for (_, row), tags in zip(batch.iterrows(), tags_list):
            rows.append({
                "image_id": row["image_id"],
                "caption": row["caption"],
                **{k: json.dumps(v, ensure_ascii=False) for k, v in tags.items()},
            })

        existing_df = pd.concat([existing_df, pd.DataFrame(rows)], ignore_index=True)
        existing_df.to_csv(output_csv, index=False)

        done = min(start + batch_size, total)
        elapsed = time.perf_counter() - t0
        print(f"[{done}/{total}] last={batch.iloc[-1]['image_id']} | elapsed={elapsed:.1f}s")

    print(f"\nDone. Saved {len(existing_df)} rows to {output_csv}")
    return existing_df

### Run batch extraction

In [13]:
# Paths come from the Configuration cell. Choose your run settings below.
results_df = batch_extraction(
    captions_csv=CAPTIONS_CSV,
    output_csv=OUTPUT_CSV,
    model=model,
    tokenizer=tokenizer,
    batch_size=2,  # captions per batch
    limit=4,       # max pending captions for this run (None = all)
    resume=True    # skip image_ids already present in output CSV
)

Processing 4 captions (batch_size=2).

[2/4] last=HMT2 | elapsed=22.5s
[4/4] last=KSU10 | elapsed=40.0s

Done. Saved 4 rows to /content/drive/MyDrive/W2L/Projects/AISTER/w2l/step 2/outputs/tags.csv
